# Consumer loan requirement extraction Ã¢â‚¬â€ QLoRA Colab notebook

This notebook fine-tunes only the requirement-extraction model. The SDLC ranking remains the deterministic Python scorer. Compliance remains RAG-based and is excluded from this dataset.

**Do not train from `extraction_candidates.jsonl` as-is.** It is one unvalidated example. Curate multiple complete scenarios first, then create `curated_requirements.jsonl` with `review_status` set to `approved_for_training` only after your own review. Aim for at least 50 distinct scenarios for an initial project experiment; this notebook requires 20 and warns below 50. One scenario is not enough to train a useful model.

In [ ]:
!pip -q install -U "trl[peft]" bitsandbytes datasets

## Mount Drive and load reviewed examples
Upload the curated JSONL to the folder below. Never upload real customer PII; use synthetic or de-identified descriptions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DATA_PATH = Path('/content/drive/MyDrive/loan-origination/curated_requirements.jsonl')
OUTPUT_DIR = Path('/content/drive/MyDrive/loan-origination/qwen3-1.7b-loan-requirements-lora')
MODEL_ID = 'Qwen/Qwen3-1.7B'
print('Dataset path:', DATA_PATH)
print('Output path:', OUTPUT_DIR)

In [ ]:
import json
from collections import Counter

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Upload curated_requirements.jsonl to {DATA_PATH.parent}')

rows = [json.loads(line) for line in DATA_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
rows = [row for row in rows if row.get('review_status') == 'approved_for_training']
if not rows:
    raise ValueError('No reviewed rows found. Each curated row must have review_status=approved_for_training.')

scenario_ids = [row.get('scenario_id') for row in rows]
if any(not sid for sid in scenario_ids) or len(set(scenario_ids)) != len(scenario_ids):
    raise ValueError('Every JSONL row needs a unique scenario_id; keep each full scenario in one row.')
if len(rows) < 20:
    raise ValueError(f'Only {len(rows)} reviewed scenarios. Add at least 20 to run this experiment; 50+ is a better initial target.')
if len(rows) < 50:
    print(f'WARNING: {len(rows)} scenarios is a small dataset. Treat results as an experiment, not evidence of general performance.')

required_slots = {'business_goal_and_target_customers','user_roles','application_flow','eligibility_rules','documents_and_data_collected','credit_bureau_integration','approval_process_and_thresholds','rejection_reappeal_process','disbursement_method','repayment_emi_terms','authentication_and_esign','notifications','regulatory_jurisdiction','existing_system_integrations','timeline_and_budget','expected_frequency_of_change','regulatory_audit_criticality','one_shot_description','unmapped'}
valid_categories = {'business','stakeholder','functional','security','privacy','regulatory','performance','availability','usability','data-management','integration','audit-reporting','operational'}
valid_priorities = {'low','medium','high','critical'}
for row in rows:
    assert isinstance(row.get('input', {}).get('description'), str) and row['input']['description'].strip(), row['scenario_id']
    requirements = row.get('target', {}).get('requirements')
    assert isinstance(requirements, list) and requirements, f"{row['scenario_id']}: missing target requirements"
    for req in requirements:
        assert req.get('slot') in required_slots, f"{row['scenario_id']}: invalid source slot"
        assert isinstance(req.get('statement'), str) and req['statement'].strip()
        assert isinstance(req.get('category'), list) and set(req['category']) <= valid_categories
        assert req.get('priority') in valid_priorities
        assert isinstance(req.get('acceptance_criteria'), list)
        assert isinstance(req.get('confidence_score'), (int, float)) and 0 <= req['confidence_score'] <= 1
    factors = row.get('target', {}).get('sdlc_factor_inputs', {})
    assert isinstance(factors, dict), f"{row['scenario_id']}: sdlc_factor_inputs must be an object"

print('Reviewed scenarios:', len(rows))
print('Requirement counts:', Counter(len(row['target']['requirements']) for row in rows))

## Create scenario-level train and validation split
All requirements from one client description stay together in one split, which helps prevent data leakage.

In [ ]:
from datasets import Dataset

scenario_split = Dataset.from_list(rows).train_test_split(test_size=0.2, seed=42)
train_rows = list(scenario_split['train'])
eval_rows = list(scenario_split['test'])
SYSTEM_PROMPT = '''You extract atomic, traceable software requirements from a stakeholder description for a personal/consumer loan application. Use only facts supported by the description. Do not invent thresholds, policies, regulations, integrations, or behaviors. Split distinct behaviors into separate requirements. Return only JSON with this shape: {"requirements":[{"slot":"one of the supplied source slots or one_shot_description/unmapped","statement":"The system shall ...","category":["valid category"],"priority":"low|medium|high|critical","acceptance_criteria":["observable criterion"],"confidence_score":0.0}],"sdlc_factor_inputs":{"expected_frequency_of_change":"explicit fact or empty string","regulatory_audit_criticality":"explicit fact or empty string","existing_system_integrations":"explicit fact or empty string","timeline_and_budget":"explicit fact or empty string"}}. Keep requirements concise. If a factor was not explicitly stated, use an empty string rather than inferring it.'''

def to_prompt_completion(row):
    return {
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': 'Client description:\\n' + row['input']['description']},
        ],
        'completion': [
            {'role': 'assistant', 'content': json.dumps(row['target'], ensure_ascii=False)},
        ],
    }

train_dataset = Dataset.from_list([to_prompt_completion(row) for row in train_rows])
eval_dataset = Dataset.from_list([to_prompt_completion(row) for row in eval_rows])
print('Train scenarios:', len(train_dataset), 'Validation scenarios:', len(eval_dataset))

## Load the base model in 4-bit and attach LoRA
Qwen3-1.7B is a small Apache-2.0 model option for a Colab experiment. Check the model card and license before use. A 4-bit load reduces GPU memory use; available Colab hardware still varies.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU, then rerun this cell.')
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    dtype=compute_dtype,
)
model.config.use_cache = False
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules='all-linear',
)

training_args = SFTConfig(
    output_dir='/content/loan-requirements-checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    logging_steps=5,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    max_length=4096,
    completion_only_loss=True,
    gradient_checkpointing=True,
    bf16=(compute_dtype == torch.bfloat16),
    fp16=(compute_dtype == torch.float16),
    report_to='none',
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)
trainer.train()

In [ ]:
# Inspect one held-out description; this is a qualitative check, not a metric.
sample = eval_rows[0]
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Client description:\n" + sample["input"]["description"]},
]
model.config.use_cache = True
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    enable_thinking=False,
    return_tensors="pt",
    return_dict=True,
).to(model.device)
with torch.no_grad():
    generated = trainer.model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
    )
prediction = tokenizer.decode(
    generated[0, inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
)
print(prediction)
# Strict parse check: fix the prompt/data or generation if this fails.
parsed_prediction = json.loads(prediction)
assert isinstance(parsed_prediction.get("requirements"), list)


## Save adapter and tokenizer to Drive
The output is a LoRA adapter, not a standalone full model. Keep the base model ID with it.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print('Saved adapter to:', OUTPUT_DIR)

## Next evaluation step
Run the held-out scenarios through the adapter, parse the generated JSON, and score requirement precision/recall/F1 and unsupported-requirement rate. Compare against the unfine-tuned base model. Do not integrate the adapter just because training completed; use the evaluation set and inspect failures first.